In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
单进程 Turbo 版：规范音频 -> 转写 -> 合并
为避免 Jupyter + 多进程崩溃，改为单进程高线程，速度依然快且稳定。
"""

import os, sys, subprocess
from pathlib import Path
from tqdm import tqdm

# 路径配置（按你的要求）
INPUT_DIR = Path("/Users/sagawithme/Downloads/tt/hofvtruth")
AUDIO_DIR = Path("/Users/sagawithme/Downloads/tt/audio")
TRANS_DIR = Path("/Users/sagawithme/Downloads/tt/transcripts")
MERGED_TXT = Path("/Users/sagawithme/Downloads/tt/all_transcripts.txt")

VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".webm", ".m4v", ".avi", ".ts"}
AUDIO_EXTS = {".wav", ".m4a", ".mp3", ".aac", ".flac", ".ogg", ".opus"}

# 性能参数 —— 单进程高线程
MODEL_NAME = "medium"  # 速度/准确度平衡；最准用 "large-v3"，最快用 "small"
FORCE_LANG = "en"  # 主要英文设 "en"；主要中文设 "zh"；混合/不确定设 None
VAD_FILTER = True  # 音频很干净可改 False 提速
BEAM_SIZE = 1  # 1=贪婪（最快）；>1 更准但更慢
CPU_THREADS = max(4, (os.cpu_count() or 8) - 1)  # 基本吃满
NUM_WORKERS = 2  # 2~3 即可；再高收益不大

# 避免底层库争抢线程
os.environ.setdefault("CT2_USE_EXCLUSIVE_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", str(CPU_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMexpr_NUM_THREADS", "1")


def ensure_ffmpeg():
    try:
        subprocess.run(
            ["ffmpeg", "-version"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=True,
        )
        return True
    except Exception:
        return False


def collect_sources(root: Path):
    files = []
    for p, _, names in os.walk(root):
        for nm in names:
            if nm.startswith("."):  # 跳隐藏文件
                continue
            f = Path(p) / nm
            suf = f.suffix.lower()
            if suf in VIDEO_EXTS or suf in AUDIO_EXTS:
                files.append(f)
    files.sort()
    return files


def to_wav(src: Path, dst_dir: Path) -> Path | None:
    dst_dir.mkdir(parents=True, exist_ok=True)
    out_wav = dst_dir / f"{src.stem}.wav"
    if out_wav.exists():
        return out_wav
    cmd = [
        "ffmpeg",
        "-y",
        "-i",
        str(src),
        "-vn",
        "-ac",
        "1",
        "-ar",
        "16000",
        "-c:a",
        "pcm_s16le",
        str(out_wav),
    ]
    try:
        subprocess.run(
            cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True
        )
        return out_wav if out_wav.exists() else None
    except Exception as e:
        print(f"[WARN] 转码失败：{src} -> {e}")
        return None


def merge_transcripts(in_dir: Path, merged_path: Path):
    txts = sorted(in_dir.glob("*.txt"))
    merged_path.parent.mkdir(parents=True, exist_ok=True)
    with open(merged_path, "w", encoding="utf-8") as out:
        for tp in txts:
            out.write(f"===== {tp.stem} =====\n")
            try:
                out.write(tp.read_text(encoding="utf-8", errors="ignore").strip())
            except Exception:
                out.write("[读取失败]")
            out.write("\n\n")
    print(f"已生成合并文本：{merged_path}")


def main():
    if not INPUT_DIR.exists():
        print(f"输入目录不存在：{INPUT_DIR}")
        sys.exit(1)
    if not ensure_ffmpeg():
        print("未检测到 ffmpeg（macOS: brew install ffmpeg）")
        sys.exit(1)

    # 收集源文件
    sources = collect_sources(INPUT_DIR)
    if not sources:
        print(f"未在 {INPUT_DIR} 找到可处理的媒体文件。")
        sys.exit(0)

    # 1) 规范音频
    wavs = []
    for src in tqdm(sources, desc="Normalizing to 16kHz mono WAV"):
        w = to_wav(src, AUDIO_DIR)
        if w:
            wavs.append(w)
    if not wavs:
        print("未生成任何 WAV 文件。")
        sys.exit(1)

    # 2) 单进程高线程转写
    TRANS_DIR.mkdir(parents=True, exist_ok=True)
    try:
        from faster_whisper import WhisperModel
    except Exception:
        print(
            "缺少 faster-whisper：请在当前环境执行  python -m pip install faster-whisper"
        )
        sys.exit(1)

    model = WhisperModel(
        MODEL_NAME,
        device="cpu",  # Apple 芯片：CPU + int8 最稳
        compute_type="int8",
        cpu_threads=CPU_THREADS,
        num_workers=NUM_WORKERS,
    )

    for wav in tqdm(wavs, desc="Transcribing"):
        stem = wav.stem
        out_txt = TRANS_DIR / f"{stem}.txt"
        if out_txt.exists():
            continue
        try:
            segments, info = model.transcribe(
                str(wav),
                vad_filter=VAD_FILTER,
                beam_size=BEAM_SIZE,
                language=FORCE_LANG,
                temperature=0.0,
                condition_on_previous_text=False,
            )
            with open(out_txt, "w", encoding="utf-8") as f:
                for seg in segments:
                    t = (seg.text or "").strip()
                    if t:
                        f.write(t + "\n")
        except Exception as e:
            with open(out_txt, "w", encoding="utf-8") as f:
                f.write(f"[ERROR] {e}\n")

    # 3) 合并
    merge_transcripts(TRANS_DIR, MERGED_TXT)

    print("\n完成：")
    print(f"  规范音频目录：{AUDIO_DIR}")
    print(f"  单条转写目录：{TRANS_DIR}")
    print(f"  合并文本文件：{MERGED_TXT}")
    print(
        f"  参数：MODEL={MODEL_NAME} | CPU_THREADS={CPU_THREADS} | NUM_WORKERS={NUM_WORKERS} | VAD={VAD_FILTER} | LANG={FORCE_LANG}"
    )


if __name__ == "__main__":
    main()

输入目录不存在：/Users/sagawithme/Downloads/tt/hofvtruth


SystemExit: 1

/Users/sagawithme/.conda/envs/IRG/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
